# M3GNet XAS pipeline showcase

This notebook displays the M3GNet encoder and XAS heads, then trains and evaluates the pipeline.

## Prerequisites

From the repository root, run:

```bash
bash tutorial_omnixas/download_omnixas_raw_data.sh
export OMNIXAS_DATA_ROOT="$HOME/OmniXAS_data"
```

The script downloads and extracts FEFF data by default. VASP download is not needed for this FEFF pipeline.

The shell script requires `curl`, `md5sum`, and `tar`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py').is_file():
    REPO_ROOT = REPO_ROOT.parent
SCRIPT = REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py'
from omnixas.model.m3gnet_xas import HEAD_HIDDEN_DIMS, FEATURE_SCALE, SPECTRUM_DIM, M3GNetXAS, XASSpectralHead

model = M3GNetXAS()
head = XASSpectralHead()
print(model)
print(f"M3GNetXAS parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"XASSpectralHead parameters: {sum(p.numel() for p in head.parameters()):,}")
print({'head_hidden_dims': HEAD_HIDDEN_DIMS, 'feature_scale': FEATURE_SCALE, 'target_dim': SPECTRUM_DIM})

In [ ]:
RUN_NAME = 'm3gnet_xas_seed42'
OUTPUT_ROOT = REPO_ROOT / 'output/training/m3gnet_xas_pipeline'
RUN_DIR = OUTPUT_ROOT / RUN_NAME

## Train the pipeline

The training script checks the input data before training. The command can take a long time.

In [ ]:
import runpy

script_args = [
    str(SCRIPT),
    '--output-root',
    str(OUTPUT_ROOT),
    '--run-name',
    RUN_NAME,
    '--gpu',
    '0',
    '--encoder-rows-per-element',
    '64',
    '--num-workers',
    '16',
    '--batch-size',
    '4096',
]
previous_argv = sys.argv
sys.argv = script_args
try:
    runpy.run_path(str(SCRIPT), run_name='__main__')
finally:
    sys.argv = previous_argv

## Evaluate the trained pipeline

In [ ]:
subprocess.run([
    sys.executable,
    str(SCRIPT),
    '--output-root',
    str(OUTPUT_ROOT),
    '--run-name',
    RUN_NAME,
    '--evaluate',
], cwd=REPO_ROOT, check=True)

## Plot validation metrics

This cell reads the generated CSV files and shows the validation metric.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics = pd.read_csv(RUN_DIR / 'tuned_validation.csv')
metrics.plot.bar(x='dataset', y='eta', legend=False, title='Tuned UniversalXAS validation eta')
plt.tight_layout()